In [5]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

## Create the function and register it

In [6]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()


def power3(double_value: float):
    return double_value ** 3

# Register for Python calls
power3_udf = udf(power3)

# Register the UDF as a SQL function
spark.udf.register("power3_udf", power3, DoubleType())


<function __main__.power3(double_value: float)>

## Use the function

In [7]:
from pyspark.sql.functions import col

df = spark.range(0, 10).toDF("num")
df.createOrReplaceTempView("numbers")
df = df.select("num", power3_udf(col("num")).alias("pow3"))

df.show()

+---+----+
|num|pow3|
+---+----+
|  0|   0|
|  1|   1|
|  2|   8|
|  3|  27|
|  4|  64|
|  5| 125|
|  6| 216|
|  7| 343|
|  8| 512|
|  9| 729|
+---+----+



In [8]:
spark.sql("""
select 
    num, 
    power3_udf(cast(num as Double)) as pow3 
from numbers
""").show()

+---+-----+
|num| pow3|
+---+-----+
|  0|  0.0|
|  1|  1.0|
|  2|  8.0|
|  3| 27.0|
|  4| 64.0|
|  5|125.0|
|  6|216.0|
|  7|343.0|
|  8|512.0|
|  9|729.0|
+---+-----+

